##### DBTC-70

In [0]:
%sql
UPDATE `p-cdh`.derivativeactive.dexposure 
SET objectivestatus =  objectivestatuscode
WHERE objectivestatus IS NULL
AND objectivestatuscode IS NOT NULL

In [0]:
%sql
UPDATE `p-cdh`.derivativearchive.dexposure 
SET objectivestatus =  objectivestatuscode
WHERE objectivestatus IS NULL
AND objectivestatuscode IS NOT NULL

#####PEE Fields Update in Archive

In [0]:
%sql
MERGE INTO `p-cdh`.primaryarchive.exposure dex
USING `d-cdh`.reference.ModificationsPEE df 
ON df.ExposureKey = dex.ExposureKey
WHEN MATCHED THEN UPDATE SET
dex.RestructuringPEEDate = df.LastRestructureDate
,dex.RestructuringPEEType = df.RestructuringPEEType
,dex.RestructuringPEETypeCode = df.PEECode

In [0]:
%sql
MERGE INTO `p-cdh`.derivativearchive.dexposure dex
USING `d-cdh`.reference.ModificationsPEE df 
ON df.ExposureKey = dex.ExposureKey
WHEN MATCHED THEN UPDATE SET
dex.RestructuringPEEDate = df.LastRestructureDate
,dex.RestructuringPEEType = df.RestructuringPEEType
,dex.RestructuringPEETypeCode = df.PEECode

#####REPBTC-62

In [0]:
dates_df = spark.sql("""
    SELECT DISTINCT CAST(BusinessDate AS STRING) as BusinessDate
    FROM `p-cdh`.derivativeactive.DExposure
    order by businessdate asc
""")
dates = [row['BusinessDate'] for row in dates_df.collect()]

In [0]:
for run_date in dates:
    exposure = spark.sql(f'''
                        SELECT * 
                        FROM `p-cdh`.primary.Exposure
                         WHERE BusinessDate = {run_date!r}
                         and exposurekey != 'EUC40000003959'

    ''')

    exposure.createOrReplaceTempView("exposure")
    exposure_profits = exposure.filter((exposure.SourceSystem == 'profits') & (exposure.PrimaryPartyFlag == True))
    exposure_profits.createOrReplaceTempView("exposure_profits") 
    # display(exposure_profits)

    exposure_qc = exposure.filter((exposure.PrimaryPartyFlag== True) & (exposure.SourceSystem == 'qc')) 
    exposure_qca = exposure.filter((exposure.PartyRelationshipTypeCode == '5000') & (exposure.SourceSystem == 'qca')) 
    exposure_qc.createOrReplaceTempView("exposure_qc")
    exposure_qca.createOrReplaceTempView("exposure_qca")

    product_qc = spark.sql(f""" select
        ProductCode,
        productkey,
        SourceSystem,
        ProductDescription,
        BusinessDate 
    FROM `p-cdh`.primary.Product
    WHERE BusinessDate = {run_date!r} and 
    SourceSystem ='qc'
    """) 
    product_qc.createOrReplaceTempView("product_qc")

    exposure_product_qc=spark.sql('''
                            select distinct  ex.ExposureKey as exposurekeyqcqca,p.ProductDescription as SubProductDescription
                            from exposure_qc ex
                            inner  join product_qc p
                            on p.productkey=ex.productkey
                            ''')

    exposure_product_qc.createOrReplaceTempView("exposure_product_qc")

    product_qca = spark.sql(f""" select
        ProductCode,
        productkey,
        SourceSystem,
        ProductDescription,
        BusinessDate 
    FROM `p-cdh`.primary.Product
    WHERE BusinessDate = {run_date!r} and
     SourceSystem ='qca'
    """) 
    product_qca.createOrReplaceTempView("product_qca")

    exposure_product_qca=spark.sql('''
                            select distinct  ex.ExposureKey as exposurekeyqcqca,p.ProductDescription as SubProductDescription
                            from exposure_qca ex
                            inner  join product_qca p
                            on p.productkey=ex.productkey
                            ''')

    exposure_product_qca.createOrReplaceTempView("exposure_product_qca")
    exposure_product_qc_qca=spark.sql('''
                            select distinct *
                            from
                            (
                            select *
                            from exposure_product_qc
                            union
                            select *
                            from exposure_product_qca
                            )
                            ''')

    exposure_product_qc_qca.createOrReplaceTempView("exposure_product_qc_qca")

    from pyspark.sql.functions import substring

    exposure_profits_exposure_product_qc_qca_join_condition = (
        substring(exposure_profits["ExposureKey"], 4, 50) == exposure_product_qc_qca["exposurekeyqcqca"]
    )

    target_table_profits = exposure_profits.join(
            exposure_product_qc_qca,
            exposure_profits_exposure_product_qc_qca_join_condition,
            "left",
        ) 	 

    (target_table_profits         
       .select("ExposureKey", "BusinessDate", "SubProductDescription")
       .createOrReplaceTempView("fix_desc")     
    )

    spark.sql(f"""
    MERGE INTO `p-cdh`.derivativeactive.dexposure      AS t
    USING      fix_desc                                AS s
    ON  t.ExposureKey  = s.ExposureKey
    AND t.BusinessDate = s.BusinessDate               
    WHEN MATCHED THEN
         UPDATE SET t.SubProductDescription = s.SubProductDescription
    """)

    print("SubProductDescription updated.")

#####QA historical view for Interaction

In [0]:
%sql
create or replace view `qa-cdh`.primaryarchive.interaction as 
select * from `p-cdh`.primaryarchive.interaction

In [0]:
%sql
create or replace view `qa-cdh`.primary.interaction as 
SELECT
  *
FROM
  `qa-cdh`.primaryarchive.interaction
WHERE
  BusinessDate < (
    SELECT
      MIN(BusinessDate) as start_date
    FROM
      `qa-cdh`.primaryactive.interaction
  )
UNION ALL
SELECT
  *
FROM
  `qa-cdh`.primaryactive.interaction

###Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('qa-cdh')
      AND table_schema in ('derivativearchive','primaryarchive')
      AND table_type='VIEW'
""")

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    full_view_name = f"`{table_catalog}`.{table_schema}.{table_name}"

    try:
        describe_full_view_name = spark.sql(f"""
            DESCRIBE EXTENDED {full_view_name}
        """)

        describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

        describe_query = spark.sql("""
            SELECT data_type
            FROM describe_full_view_name 
            WHERE col_name = 'View Text'
        """)

        sql_view_creation = describe_query.collect()[0][0]

        create_or_replace_sql = f"""
            CREATE OR REPLACE VIEW {full_view_name} AS 
            {sql_view_creation}
        """

        spark.sql(create_or_replace_sql)

        print(f"Successfully refreshed view: {full_view_name}")

    except Exception as e:
        print(f"   Failed to refresh view: {full_view_name}")


In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('qa-cdh')
      AND table_schema in ('derivativeactive','primaryactive')
      AND table_type='VIEW'
""")

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    full_view_name = f"`{table_catalog}`.{table_schema}.{table_name}"

    try:
        describe_full_view_name = spark.sql(f"""
            DESCRIBE EXTENDED {full_view_name}
        """)

        describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

        describe_query = spark.sql("""
            SELECT data_type
            FROM describe_full_view_name 
            WHERE col_name = 'View Text'
        """)

        sql_view_creation = describe_query.collect()[0][0]

        create_or_replace_sql = f"""
            CREATE OR REPLACE VIEW {full_view_name} AS 
            {sql_view_creation}
        """

        spark.sql(create_or_replace_sql)

        print(f"Successfully refreshed view: {full_view_name}")

    except Exception as e:
        print(f"   Failed to refresh view: {full_view_name}")


In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('qa-cdh')
      AND table_schema in ('derivative', 'primary')
      AND table_type = 'VIEW'
""")

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    full_view_name = f"`{table_catalog}`.{table_schema}.{table_name}"

    try:
        describe_full_view_name = spark.sql(f"""
            DESCRIBE EXTENDED {full_view_name}
        """)
        
        describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

        describe_query = spark.sql("""
            SELECT data_type
            FROM describe_full_view_name 
            WHERE col_name = 'View Text'
        """)

        sql_view_creation = describe_query.collect()[0][0]

        create_or_replace_sql = f"""
            CREATE OR REPLACE VIEW {full_view_name} AS 
            {sql_view_creation}
        """

        spark.sql(create_or_replace_sql)

        print(f" Refreshed view: {full_view_name}")

    except Exception as e:
        print(f" Failed to refresh view: {full_view_name}")
        print(e)

###Closed loans function update

In [0]:
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `p-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0
AND coalesce(B.JustificationCode,0) != 43023

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')

## Regulatory 

In [0]:
%sql
create table  `p-cdh`.reference.reg_lookupvalues as
select * from `qa-cdh`.reference.reg_lookupvalues;

In [0]:
%sql
create table  `p-cdh`.reference.reg_portfoliodefaultvalues as
select * from `qa-cdh`.reference.reg_portfoliodefaultvalues;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_cosmos_business_plan as
select * from `qa-reportinputfiles`.regulatory.reg_cosmos_business_plan;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_cosmos_waterfall as
select * from `qa-reportinputfiles`.regulatory.reg_cosmos_waterfall;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_domus_business_plan as
select * from `qa-reportinputfiles`.regulatory.reg_domus_business_plan;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_domus_waterfall as
select * from `qa-reportinputfiles`.regulatory.reg_domus_waterfall;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_rhodium_business_plan as
select * from `qa-reportinputfiles`.regulatory.reg_rhodium_business_plan;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_rhodium_waterfall as
select * from `qa-reportinputfiles`.regulatory.reg_rhodium_waterfall;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_galaxy_spv_i_business_plan as
select * from `qa-reportinputfiles`.regulatory.reg_galaxy_spv_i_business_plan;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_galaxy_spv_i_waterfall as
select * from `qa-reportinputfiles`.regulatory.reg_galaxy_spv_i_waterfall;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_galaxy_spv_ii_business_plan as
select * from `qa-reportinputfiles`.regulatory.reg_galaxy_spv_ii_business_plan;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_galaxy_spv_ii_waterfall as
select * from `qa-reportinputfiles`.regulatory.reg_galaxy_spv_ii_waterfall;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_galaxy_spv_iv_business_plan as
select * from `qa-reportinputfiles`.regulatory.reg_galaxy_spv_iv_business_plan;

In [0]:
%sql
create table  `p-reportinputfiles`.regulatory.reg_galaxy_spv_iv_waterfall as
select * from `qa-reportinputfiles`.regulatory.reg_galaxy_spv_iv_waterfall;

In [0]:
%sql
insert into `p-cdh`.workflow.externalfilesconfiguration
select 'Regulatory' as RelativePath, 'p-cdh' as UnityCatalog, 'reference' as UnitySchema, 'reg_lookupvalues' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, 'LookupCategory,LookupSubCategory,AdditionalInfo,COR26Segment,LookupType,ESMASegment,ESMAGuarantorType' as string_columns
union all
select 'Regulatory' as RelativePath, 'p-cdh' as UnityCatalog, 'reference' as UnitySchema, 'reg_portfoliodefaultvalues' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, 'PortfolioName,LEI,SPV,IsInflowToCepal,GlobalPortfolioname,OriginalLenderLEI,OriginalLenderCountry,OriginatorName,OriginatorLEI,OriginatorCountry' as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_cosmos_business_plan' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_domus_business_plan' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_rhodium_business_plan' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_galaxy_spv_i_business_plan' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_galaxy_spv_ii_business_plan' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_galaxy_spv_iv_business_plan' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all 
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_cosmos_waterfall' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_domus_waterfall' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_rhodium_waterfall' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_galaxy_spv_i__waterfall' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_galaxy_spv_ii_waterfall' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns
union all
select 'Regulatory' as RelativePath, 'p-reportinputfiles' as UnityCatalog, 'regulatory' as UnitySchema, 'reg_galaxy_spv_iv__waterfall' as TableName, 'Incremental' as `Flow`, null as ValidatorScript,
null as NotificationEmails, null as string_columns